# Customers

In [ ]:
# Imports & settings
from imports import *
notebook_settings() 

# Load Data
location_ids_by_coverage = load_loc_ids()
locations, before_after_details_true, items_tagged, customers = load_static()
sales_and_menu_data = load_sales()

### Customer Stats

Merge in customer data

In [ ]:
# Initialize dict all data
# Note: sales_and_menu_data, customers loaded via load_data_to_globals()
sales_menu_customers_data = {}
for loc_id, df in sales_and_menu_data.items():

    # VLZX7K2M9QD4T doesn't have customers
    if loc_id == 'VLZX7K2M9QD4T':
        continue
    
    # Double check customers are unique
    customers = (
        customers
        .dropna(subset=['customer_id'])
        .drop_duplicates(subset=['location_id', 'customer_id']))

    # Combine
    merged = (
        pd.merge(
            df.reset_index(), # Keep the index, since merges don't keep it 
            customers, 
            on=['location_id', 'customer_id'], how='left')
        .set_index('created_at', drop=False))
    
    # Save
    sales_menu_customers_data[loc_id] = merged

In [ ]:
def find_customers_before_after(option, df, exposure):
    
    customer_times_group = (
        df
        .drop(columns='created_at')
        .reset_index()
        .groupby('customer_id', observed=True)
        ['created_at'])
    
    if option == 'unbounded':
    
        date_bounds = (
            customer_times_group
            .agg(['min', 'max'])
            .assign(min = lambda df: df['min'].dt.tz_localize(None),
                    max = lambda df: df['max'].dt.tz_localize(None)))
        
        has_before_entry = date_bounds['min'].lt(exposure)
        has_after_entry = date_bounds['max'].gt(exposure)
        
    if option == 'bounded':
    
        date_bounds = (
            customer_times_group
            .agg(list)
            .apply(lambda dates: [d.tz_localize(None) for d in dates]))

        before, after = exposure - DateOffset(months=2), exposure + DateOffset(months=2)
        has_before_entry = date_bounds.apply(lambda dates: any(before <= d < exposure for d in dates))
        has_after_entry = date_bounds.apply(lambda dates: any(exposure < d <= after for d in dates))

    customer_indicators = has_before_entry & has_after_entry
    customers = date_bounds[customer_indicators].index

    return customer_indicators, customers


def find_customers_before_after_df(
    option = 'unbounded',
    location_ids_by_coverage=location_ids_by_coverage, 
    sales_menu_customers_data=sales_menu_customers_data,
    before_after_details_true=before_after_details_true):
    
    customers_before_after_list = []
    for loc_id in location_ids_by_coverage:
        
        if loc_id == 'VLZX7K2M9QD4T':
            continue

        df = sales_menu_customers_data[loc_id]
        promo_datetime = before_after_details_true.loc[loc_id,'cross_over_date'].tz_convert(None)
        customer_indicators, customers = find_customers_before_after(option, df, promo_datetime)
        
        # Filter main df for relevant customers
        gender_known_pct = (
            df
            .assign(created_at = lambda df: df.created_at.dt.tz_localize(None))
            .query('customer_id.isin(@customers)')
            [['customer_id','gender']]
            .drop_duplicates()
            ['gender']
            .notna()
            .mean())
        
        customers_before_after_list.append({
            'loc_id': loc_id,
            'total w/ before + after': customer_indicators.sum(),
            'pct w/ before + after': customer_indicators.mean(),
            'pct w/ before + after w/ gender': gender_known_pct,
            'customer_ids': customers.tolist()})
        
    customers_before_after = pd.DataFrame(customers_before_after_list)

    return customers_before_after

customers_before_after = find_customers_before_after_df('unbounded')
display(customers_before_after.style.format(precision=2))
customers_before_after.to_pickle('data/customers_before_after.pkl')

customers_before_after_intervention = find_customers_before_after_df('bounded')
display(customers_before_after_intervention.style.format(precision=2))
customers_before_after_intervention.to_pickle('data/customers_before_after_intervention.pkl')

Totals

In [ ]:
# Number of customers, number of customers with gender data, number of customers with age data
print(f'Total number of customers: {customers.shape[0]:,}')
print(f'Customers with gender data: {customers["gender"].notna().sum():,}')
print(f'Customers with age data: {customers["age"].notna().sum():,}')

Percent Customer IDs

In [ ]:
rows = []
stats = dict(n_id=0, t_rows=0, n_id_txn=0, t_txn=0)
for loc_id in location_ids_by_coverage:
    
    if loc_id == 'VLZX7K2M9QD4T':
        continue
    
    # Entries
    df = sales_menu_customers_data[loc_id].drop(columns='created_at')
    n_id, t_rows = df['customer_id'].notna().sum(), df.shape[0]
    p_id = n_id / t_rows
    
    # Transactions
    has_id = df.groupby('order_id', observed=True)['customer_id'].nunique() # binary column since multiple customers can't be under one order
    n_id_txn, t_txn = has_id.sum(), has_id.size
    p_id_txn = n_id_txn / t_txn
    
    # Store
    rows.append(dict(location=loc_id, n_id=n_id, t_rows=t_rows, p_id=p_id, txn_cust=n_id_txn, t_txn=t_txn, p_id_txn=p_id_txn))

customer_id_percents = (
    pd.DataFrame(rows)
    .style.format(precision=2)
    .pipe(lambda df: pd.concat([
        df,
        pd.DataFrame([{
            'location': 'TOTAL',
            'n_id': df['n_id'].sum(),
            't_rows': df['t_rows'].sum(),
            'p_id': df['n_id'].sum() / df['t_rows'].sum(),
            'txn_cust': df['txn_cust'].sum(),
            't_txn': df['t_txn'].sum(),
            'p_id_txn': df['txn_cust'].sum() / df['t_txn'].sum()}])], ignore_index=True)))

display(customer_id_percents)

Intersecting customers bases

In [ ]:
# Precompute the set of customers in each restaurant for efficiency
precomputed_customers = {}
for loc_id in location_ids_by_coverage:
    precomputed_customers[loc_id] = set(customers.query('location_id == @loc_id')['customer_id'].unique().tolist())

# Restaurant 1
for i, loc_id1 in enumerate(location_ids_by_coverage):
    customer_set1 = precomputed_customers[loc_id1]

    # Restaurant 2
    for loc_id2 in location_ids_by_coverage[i:]:
        
        # As long as they're different
        if loc_id1 != loc_id2:
            customer_set2 = precomputed_customers[loc_id2]

            # Intersect
            intersection = customer_set1.intersection(customer_set2)

            # Result
            if intersection:
                print(loc_id1, loc_id2, len(intersection))

Gender Proportions

In [ ]:
# Initialize female proportions list
female_proportions_list = []
for loc_id in location_ids_by_coverage:
    
    if loc_id == 'VLZX7K2M9QD4T':
        continue
    
    df = sales_menu_customers_data[loc_id]
    print(loc_id)
    # Prevent overwriting
    df = df.copy()

    # Initialize summary row for resulting summary dataframe
    row = {'location_id': loc_id}

    # Promo date (normalize time zones)
    cross_over = before_after_details_true.loc[loc_id,'cross_over_date']
    #df.index = df.index.tz_localize(None)

    # Subset and aggregate the get the number of customers of genders
    before_genders = df.loc[:cross_over,'gender'].value_counts()
    after_genders = df.loc[cross_over:,'gender'].value_counts()
    
    
    # As long as they're nonempty, calculate the fractions
    if not before_genders.empty and not after_genders.empty:
        before_female_total = before_genders.loc['female']
        after_female_total = after_genders.loc['female']
        before_known_gender_total = before_genders.loc['male'] + before_genders.loc['female']
        after_known_gender_total = after_genders.loc['male'] + after_genders.loc['female']

        # No zero divisor allowed
        before_frac_female = -1
        if before_known_gender_total != 0:
            before_frac_female = before_female_total/before_known_gender_total
            
        # No zero divisor allowed
        after_frac_female = -1
        if after_known_gender_total != 0:
            after_frac_female = after_female_total/after_known_gender_total
        
        # Is it a large enough sample?
        large_enough = 1000
        sample_qualifer = ""
        if large_enough < before_known_gender_total and large_enough < after_known_gender_total:
            sample_qualifer = "Large Enough Sample"

        # Store in summary row
        row['b_f_frac'] = round(before_frac_female*100)/100
        row['a_f_frac'] = round(after_frac_female*100)/100
        row['enough_data'] = bool(sample_qualifer)
        row['b_notna_total'] = before_known_gender_total
        row['a_notna_total'] = after_known_gender_total


    # If both are empty, there's no data
    elif before_genders.empty and after_genders.empty:

        print(loc_id, "--No customer data!")

    # If one is empty, there's no comparison
    else:

        print(loc_id, "--Not enough data before or after.")

    # Save
    female_proportions_list.append(row)

female_proportions = pd.DataFrame(female_proportions_list)
display(female_proportions)

Revisiting customers

In [ ]:
def retrieve_ids(df, ids, j):
    if j == 10:
        ids.extend(df.index.tolist())
    return df

# Initialize summary list for revisiting customers
customer_revisit_row_list = []
dict_of_customer_ids = {}
for loc_id, df in sales_menu_customers_data.items():
    
    # Summary row
    row = {'location_id': loc_id, 'total': df['customer_id'].nunique()}

    # Did they revisit _x_ number of times?
    revisit_times = [1, 2, 5, 10]
    customer_ids = []
    for j in revisit_times:

        # For each customer, did they come at multiple times
        num_revisits = (df
                        .groupby('customer_id', observed=True)
                        ['created_at']
                        .nunique()
                        .to_frame('revisits')
                        .query('revisits > @j')
                        .pipe(retrieve_ids, customer_ids, j)
                        .shape[0])

        # Store in summary row
        row['more than ' + str(j)] = num_revisits

    # Save
    customer_revisit_row_list.append(row)

    dict_of_customer_ids[loc_id] = customer_ids
    
revisits = pd.DataFrame(customer_revisit_row_list)
display(revisits)

Revisiting Customers Visual

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))

for loc_id in location_ids_by_coverage[:1]:
    
    print(loc_id)
    df = sales_and_menu_data[loc_id]
    customers = dict_of_customer_ids[loc_id]
    promo_datetime = before_after_details_true.loc[loc_id,'cross_over_date'].tz_convert(None)
    
    customer_active_weeks_dict = {}
    for customer in customers[:60]:
        customer_active_weeks_dict[customer] = (df
                                                .query('customer_id == @ customer')
                                                .resample('W')
                                                .size()
                                                .to_frame(name='W')
                                                .query('0 < W')
                                                .index
                                                .tz_localize(None)
                                                .to_period('W')
                                                .tolist())
    
    for customer, active_weeks in customer_active_weeks_dict.items():
        
        # For every active week
        for week in active_weeks:

            # Place a blue dot
            ax.hlines(y=customer, xmin=week.start_time, xmax=week.end_time, colors='blue', lw=2, label=customer)

        # Place a red circle for the promotional item
        ax.plot(promo_datetime, customer, 'ro', alpha=0.5)
    
    ax.set_title(loc_id)